# HW3: Image Classification with CNN

Task: classify images with CNN models.

- Only competition data may be used.
- Models must be trained from scratch; no pretrained weights.
- Training images: `train/x_y.png`, where `x` is the category.
- Validation images: `valid/x_y.png`, where `x` is the category.
- Testing images: `test/n.png`, where `n` is the id.

## Download the Dataset

In [ ]:
# !pip install kagglehub==1.0.1

import inspect
import os
from pathlib import Path
from zipfile import ZipFile

DEFAULT_DOWNLOAD_DIR = Path("machine-learning/hung-yi-lee-machine-learning/HW3")
if not DEFAULT_DOWNLOAD_DIR.exists():
    DEFAULT_DOWNLOAD_DIR = Path.cwd()
DEFAULT_DOWNLOAD_DIR = DEFAULT_DOWNLOAD_DIR.resolve()
DATA_DOWNLOAD_DIR = DEFAULT_DOWNLOAD_DIR / "data"
DATA_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Optional: set this to a local dataset directory if the competition files were downloaded manually.
DATA_DOWNLOAD_ROOT = None
# DATA_DOWNLOAD_ROOT = "/path/to/ml2023spring-hw3"

# Optional: set this if the token is not under the current user's home directory.
KAGGLE_CREDENTIALS_DIR = None
# KAGGLE_CREDENTIALS_DIR = "/path/to/.kaggle"

KAGGLE_CONFIG_DIR = Path(
    KAGGLE_CREDENTIALS_DIR
    or os.environ.get("KAGGLE_CONFIG_DIR", Path.home() / ".kaggle")
).expanduser().resolve()
KAGGLE_ACCESS_TOKEN = KAGGLE_CONFIG_DIR / "access_token"
KAGGLE_JSON = KAGGLE_CONFIG_DIR / "kaggle.json"
os.environ["KAGGLE_CONFIG_DIR"] = str(KAGGLE_CONFIG_DIR)
os.environ["KAGGLEHUB_CACHE"] = str(DATA_DOWNLOAD_DIR)


def find_existing_data_dir(root: Path) -> Path | None:
    candidates = [root]
    if root.exists():
        candidates.extend(candidate for candidate in root.glob("**/*") if candidate.is_dir())

    for candidate in candidates:
        if (candidate / "train").is_dir() and (candidate / "valid").is_dir() and (candidate / "test").is_dir():
            return candidate.resolve()
    return None

def has_kaggle_credentials() -> bool:
    has_env_credentials = bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    has_access_token = KAGGLE_ACCESS_TOKEN.exists()
    has_json_credentials = KAGGLE_JSON.exists()
    if has_access_token:
        KAGGLE_ACCESS_TOKEN.chmod(0o600)
    if has_json_credentials:
        KAGGLE_JSON.chmod(0o600)
    return has_env_credentials or has_access_token or has_json_credentials


def download_competition(handle: str, output_dir: Path) -> str:
    import kagglehub

    signature = inspect.signature(kagglehub.competition_download)
    if "output_dir" in signature.parameters:
        return kagglehub.competition_download(handle, output_dir=str(output_dir))
    return kagglehub.competition_download(handle)


if DATA_DOWNLOAD_ROOT is None:
    existing_data_dir = find_existing_data_dir(DATA_DOWNLOAD_DIR)
    if existing_data_dir is not None:
        download_path = existing_data_dir
        path = str(download_path)
    else:
        if not has_kaggle_credentials():
            raise FileNotFoundError(
                f"Kaggle credentials were not found. Put the KaggleHub token at {KAGGLE_ACCESS_TOKEN}, "
                f"put legacy API credentials at {KAGGLE_JSON}, or set KAGGLE_USERNAME and KAGGLE_KEY "
                "before starting this notebook. Do not paste the token into this notebook."
            )

        try:
            downloaded_root = Path(download_competition("ml2023spring-hw3", DATA_DOWNLOAD_DIR)).expanduser().resolve()
        except Exception as error:
            error_name = error.__class__.__name__
            error_text = str(error)
            if error_name == "UnauthenticatedError":
                raise RuntimeError(
                    "KaggleHub still cannot authenticate. Make sure the current Jupyter kernel can see "
                    f"{KAGGLE_ACCESS_TOKEN} or {KAGGLE_JSON}, or restart the kernel after setting environment credentials. "
                    "Also join the competition and accept its rules on Kaggle before re-running this cell."
                ) from error
            if "403" in error_text and "rules" in error_text.lower():
                raise PermissionError(
                    "Kaggle authentication succeeded, but this account does not have permission to download "
                    "the competition data yet. Open https://www.kaggle.com/competitions/ml2023spring-hw3/rules, "
                    "join the competition, accept the rules, then re-run this cell."
                ) from error
            raise

        for zip_path in downloaded_root.glob("*.zip"):
            target_dir = downloaded_root / zip_path.stem
            if not target_dir.exists():
                with ZipFile(zip_path) as archive:
                    archive.extractall(target_dir)

        download_path = find_existing_data_dir(downloaded_root) or find_existing_data_dir(DATA_DOWNLOAD_DIR)
        if download_path is None:
            raise FileNotFoundError("Could not find the downloaded image data directory.")
        path = str(download_path)
else:
    download_path = Path(DATA_DOWNLOAD_ROOT).expanduser().resolve()
    path = str(download_path)

print("Data directory:", download_path)
print("KaggleHub cache:", os.environ["KAGGLEHUB_CACHE"])


## Setup and Model Overview

For image $X\in\mathbb{R}^{H\times W\times C}$, a CNN learns filters

$$
Y_{i,j,c}=\sigma\left(\sum_{m,n,d}X_{i+m,j+n,d}W_{m,n,d,c}+b_c\right).
$$

The classifier is trained from scratch by minimizing cross entropy:

$$
L=-\log p_y, \qquad p=\operatorname{softmax}(f_\theta(X)).
$$

In [ ]:
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm

try:
    import wandb
except ImportError:
    wandb = None


@dataclass
class Config:
    seed: int = 42
    model_name: str = "resnet50"  # resnet34, resnet50, convnext_tiny, swin_t; all use weights=None.
    image_size: int = 224
    batch_size: int = 128
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    max_epochs: int = 80
    early_stopping_patience: int = 12
    label_smoothing: float = 0.1
    gradient_clip_norm: float = 5.0
    num_workers: int = 8
    prefetch_factor: int = 2
    gpu_id: int = 0
    use_amp: bool = True
    use_tta: bool = True
    show_batch_progress: bool = False
    compile_model: bool = False
    model_filename: str = "hw3_image_classifier_best.pt"
    submission_filename: str = "submission_hw3.csv"
    use_wandb: bool = False
    wandb_project: str = "hung-yi-lee-machine-learning"
    wandb_run_name: str = "hw3-scratch-strong-augmentation"


CONFIG = Config()


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def choose_device() -> torch.device:
    if torch.cuda.is_available():
        gpu_count = torch.cuda.device_count()
        gpu_id = CONFIG.gpu_id if 0 <= CONFIG.gpu_id < gpu_count else 0
        if gpu_id != CONFIG.gpu_id:
            print(f"Requested CUDA GPU {CONFIG.gpu_id}, but only {gpu_count} GPU(s) are available. Falling back to cuda:0.")
        if hasattr(torch, "set_float32_matmul_precision"):
            torch.set_float32_matmul_precision("high")
        torch.backends.cudnn.benchmark = True
        device = torch.device(f"cuda:{gpu_id}")
        torch.cuda.set_device(device)
        print(f"Using CUDA GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}")
        return device

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("Using Apple Metal Performance Shaders (MPS).")
        return torch.device("mps")

    print("Using CPU. GPU acceleration is not available in this environment.")
    return torch.device("cpu")


def to_device(tensor: torch.Tensor) -> torch.Tensor:
    return tensor.to(device, non_blocking=(device.type == "cuda"))


def dataloader_kwargs(shuffle: bool = False) -> dict:
    kwargs = {
        "batch_size": CONFIG.batch_size,
        "shuffle": shuffle,
        "num_workers": CONFIG.num_workers,
        "pin_memory": device.type == "cuda",
    }
    if CONFIG.num_workers > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = CONFIG.prefetch_factor
    return kwargs


def init_wandb():
    if not CONFIG.use_wandb:
        return None
    if wandb is None:
        raise ImportError("Install wandb or set CONFIG.use_wandb=False.")
    return wandb.init(
        project=CONFIG.wandb_project,
        name=CONFIG.wandb_run_name,
        config=asdict(CONFIG),
    )


set_seed(CONFIG.seed)
device = choose_device()
wandb_run = init_wandb()

print(f"Using device: {device}")
print(asdict(CONFIG))


## Locate and Inspect the Dataset

In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}


def find_data_dir() -> Path:
    roots: list[Path] = [Path.cwd(), Path.cwd().parent, Path(".").resolve()]
    if "path" in globals():
        roots.append(Path(path))

    candidates: list[Path] = []
    for root in roots:
        candidates.append(root)
        if root.exists():
            candidates.extend(candidate for candidate in root.glob("**/*") if candidate.is_dir())

    for candidate in candidates:
        if (candidate / "train").is_dir() and (candidate / "valid").is_dir() and (candidate / "test").is_dir():
            return candidate.resolve()

    raise FileNotFoundError("Could not find a directory containing train, valid, and test folders.")


def list_images(directory: Path) -> list[Path]:
    return sorted(path for path in directory.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)


def label_from_path(path: Path) -> int:
    return int(path.stem.split("_", maxsplit=1)[0])


def test_id_from_path(path: Path) -> int:
    return int(path.stem)


DATA_DIR = find_data_dir()
TRAIN_DIR = DATA_DIR / "train"
VALID_DIR = DATA_DIR / "valid"
TEST_DIR = DATA_DIR / "test"

train_paths = list_images(TRAIN_DIR)
valid_paths = list_images(VALID_DIR)
test_paths = sorted(list_images(TEST_DIR), key=test_id_from_path)
class_ids = sorted({label_from_path(path) for path in train_paths + valid_paths})
class_to_index = {class_id: index for index, class_id in enumerate(class_ids)}
index_to_class = {index: class_id for class_id, index in class_to_index.items()}

print("Data directory:", DATA_DIR)
print("Train images:", len(train_paths))
print("Valid images:", len(valid_paths))
print("Test images:", len(test_paths))
print("Classes:", class_ids)

## Prepare Image Datasets

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def build_train_transform() -> transforms.Compose:
    return transforms.Compose([
        transforms.RandomResizedCrop(
            CONFIG.image_size,
            scale=(0.65, 1.0),
            ratio=(0.75, 1.33),
            interpolation=InterpolationMode.BICUBIC,
        ),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), ratio=(0.3, 3.3)),
    ])


def build_eval_transform(horizontal_flip: bool = False) -> transforms.Compose:
    transform_steps: list = [
        transforms.Resize(int(CONFIG.image_size * 1.15), interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(CONFIG.image_size),
    ]
    if horizontal_flip:
        transform_steps.append(transforms.RandomHorizontalFlip(p=1.0))
    transform_steps.extend([
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return transforms.Compose(transform_steps)


class ImageClassificationDataset(Dataset):
    def __init__(self, image_paths: list[Path], transform, with_labels: bool = True) -> None:
        self.image_paths = image_paths
        self.transform = transform
        self.with_labels = with_labels

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, index: int):
        image_path = self.image_paths[index]
        with Image.open(image_path) as image:
            x = self.transform(image.convert("RGB"))

        if self.with_labels:
            y = class_to_index[label_from_path(image_path)]
            return x, torch.tensor(y, dtype=torch.long)

        return x, torch.tensor(test_id_from_path(image_path), dtype=torch.long)


train_transform = build_train_transform()
valid_transform = build_eval_transform()
test_transforms = [build_eval_transform()]
if CONFIG.use_tta:
    test_transforms.append(build_eval_transform(horizontal_flip=True))

train_dataset = ImageClassificationDataset(train_paths, transform=train_transform, with_labels=True)
valid_dataset = ImageClassificationDataset(valid_paths, transform=valid_transform, with_labels=True)
test_dataset = ImageClassificationDataset(test_paths, transform=test_transforms[0], with_labels=False)

train_loader = DataLoader(train_dataset, **dataloader_kwargs(shuffle=True))
valid_loader = DataLoader(valid_dataset, **dataloader_kwargs())
test_loader = DataLoader(test_dataset, **dataloader_kwargs())

print("Train transform:", train_transform)
print("Eval transform:", valid_transform)
print("TTA views:", len(test_transforms))


## Build and Train the CNN

In [ ]:
def create_model(model_name: str, num_classes: int) -> nn.Module:
    model_name = model_name.lower()

    if model_name == "resnet34":
        model = models.resnet34(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if model_name == "resnet50":
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if model_name == "convnext_tiny":
        model = models.convnext_tiny(weights=None)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
        return model

    if model_name == "swin_t":
        model = models.swin_t(weights=None)
        model.head = nn.Linear(model.head.in_features, num_classes)
        return model

    raise ValueError(f"Unknown model_name: {model_name}")


model = create_model(CONFIG.model_name, num_classes=len(class_ids)).to(device)
if CONFIG.compile_model and hasattr(torch, "compile") and device.type == "cuda":
    model = torch.compile(model)

criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG.max_epochs, eta_min=CONFIG.learning_rate * 0.01)
amp_enabled = CONFIG.use_amp and device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f"Model: {CONFIG.model_name} (weights=None)")
print(f"Trainable parameters: {trainable_params:,}")
print(f"AMP enabled: {amp_enabled}")
if wandb_run is not None:
    wandb.watch(model, log="gradients", log_freq=100)
model


In [ ]:
def autocast_context():
    if amp_enabled:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def run_epoch(model: nn.Module, loader: DataLoader, training: bool, desc: str) -> tuple[float, float]:
    model.train(training)
    total_loss = 0.0
    correct = 0
    total = 0
    progress = tqdm(loader, desc=desc, leave=False, disable=not CONFIG.show_batch_progress)

    for x_batch, y_batch in progress:
        x_batch = to_device(x_batch)
        y_batch = to_device(y_batch)

        with torch.set_grad_enabled(training):
            with autocast_context():
                logits = model(x_batch)
                loss = criterion(logits, y_batch)

        if training:
            optimizer.zero_grad(set_to_none=True)
            if amp_enabled:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG.gradient_clip_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG.gradient_clip_norm)
                optimizer.step()

        total_loss += loss.item() * len(x_batch)
        correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total += len(x_batch)

    return total_loss / total, correct / total


history: list[dict[str, float]] = []
best_valid_accuracy = 0.0
epochs_without_improvement = 0
checkpoint_path = Path(CONFIG.model_filename)
epoch_bar = tqdm(range(1, CONFIG.max_epochs + 1), desc="Training", dynamic_ncols=True)

for epoch in epoch_bar:
    train_loss, train_accuracy = run_epoch(model, train_loader, training=True, desc=f"train {epoch:03d}")
    valid_loss, valid_accuracy = run_epoch(model, valid_loader, training=False, desc=f"valid {epoch:03d}")
    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    is_best = valid_accuracy > best_valid_accuracy
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "valid_loss": valid_loss,
        "valid_accuracy": valid_accuracy,
        "learning_rate": current_lr,
        "is_best": is_best,
    })

    if wandb_run is not None:
        wandb.log({
            "epoch": epoch,
            "train/loss": train_loss,
            "train/accuracy": train_accuracy,
            "valid/loss": valid_loss,
            "valid/accuracy": valid_accuracy,
            "learning_rate": current_lr,
        })

    if is_best:
        best_valid_accuracy = valid_accuracy
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": asdict(CONFIG),
            "class_ids": class_ids,
            "valid_accuracy": valid_accuracy,
            "epoch": epoch,
        }, checkpoint_path)
    else:
        epochs_without_improvement += 1

    epoch_bar.set_postfix({
        "train_acc": f"{train_accuracy:.4f}",
        "valid_acc": f"{valid_accuracy:.4f}",
        "best": f"{best_valid_accuracy:.4f}",
        "lr": f"{current_lr:.2e}",
        "stale": epochs_without_improvement,
    })

    if epochs_without_improvement >= CONFIG.early_stopping_patience:
        break

epoch_bar.close()
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Best validation accuracy: {checkpoint['valid_accuracy']:.4f} at epoch {checkpoint['epoch']}")
if wandb_run is not None:
    wandb.summary["best_valid_accuracy"] = checkpoint["valid_accuracy"]
    wandb.finish()


In [ ]:
history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["valid_loss"], label="valid")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_accuracy"], label="train")
axes[1].plot(history_df["epoch"], history_df["valid_accuracy"], label="valid")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.show()

## Predict the Test Set and Export the Submission

In [ ]:
@torch.no_grad()
def predict_with_tta(model: nn.Module, image_paths: list[Path], tta_transforms: list[transforms.Compose]) -> pd.DataFrame:
    model.eval()
    probability_sum = np.zeros((len(image_paths), len(class_ids)), dtype=np.float32)
    image_ids = np.array([test_id_from_path(path) for path in image_paths])

    for view_index, transform in enumerate(tta_transforms, start=1):
        dataset = ImageClassificationDataset(image_paths, transform=transform, with_labels=False)
        loader = DataLoader(dataset, **dataloader_kwargs())
        offset = 0

        for x_batch, _ in tqdm(loader, desc=f"TTA {view_index}/{len(tta_transforms)}", leave=False):
            x_batch = to_device(x_batch)
            with autocast_context():
                logits = model(x_batch)
            probs = torch.softmax(logits, dim=1).float().cpu().numpy()
            probability_sum[offset:offset + len(probs)] += probs
            offset += len(probs)

    predicted_indices = probability_sum.argmax(axis=1)
    rows = [
        {"Id": int(image_id), "Category": int(index_to_class[int(predicted_index)])}
        for image_id, predicted_index in zip(image_ids, predicted_indices)
    ]
    return pd.DataFrame(rows).sort_values("Id").reset_index(drop=True)


submission = predict_with_tta(model, test_paths, test_transforms)
submission_path = Path(CONFIG.submission_filename)
submission.to_csv(submission_path, index=False)

print(f"Saved submission to: {submission_path.resolve()}")
submission.head()
